# NHS A&E Data Cleaning

This notebook prepares the raw NHS A&E attendance dataset for analysis in Python, SQL, Excel and Power BI while preserving the original raw data.

In [3]:
import pandas as pd
from pathlib import Path

# File locations
raw_file = Path("../data/raw/AE_attendances_england_monthly.csv")
clean_file = Path("../data/cleaned/AE_attendances_cleaned.csv")

# Load raw dataset
df = pd.read_csv(raw_file)

print("Original shape:", df.shape)

# Work on a copy so the raw dataset remains untouched
clean_df = df.copy()

# Remove unnecessary exported index column
clean_df = clean_df.drop(columns=["Unnamed: 0"], errors="ignore")

# Standardise dates
clean_df["date"] = pd.to_datetime(clean_df["date"], errors="coerce")

# Remove exact duplicate records
duplicates_before = clean_df.duplicated().sum()
clean_df = clean_df.drop_duplicates()

# Remove records without a valid date or organisation name
clean_df = clean_df.dropna(subset=["date", "Name"])

# Remove accidental spaces from organisation names
clean_df["Name"] = clean_df["Name"].str.strip()

# Rebuild month/year from the validated date
clean_df["month"] = clean_df["date"].dt.month
clean_df["year"] = clean_df["date"].dt.year

# Flag impossible four-hour percentages instead of silently treating them as valid
performance_column = "Percentage in 4 hours or less (all)"
invalid_performance = ~clean_df[performance_column].between(0, 100)

print("Duplicate rows removed:", duplicates_before)
print("Invalid four-hour percentages:", invalid_performance.sum())

clean_df.loc[invalid_performance, performance_column] = pd.NA

# Save cleaned dataset
clean_file.parent.mkdir(parents=True, exist_ok=True)
clean_df.to_csv(clean_file, index=False)

print("Cleaned shape:", clean_df.shape)
print("Saved to:", clean_file)

Original shape: (27112, 20)
Duplicate rows removed: 0
Invalid four-hour percentages: 0
Cleaned shape: (27108, 19)
Saved to: ..\data\cleaned\AE_attendances_cleaned.csv


In [6]:
print("Rows:", len(clean_df))
print("Columns:", len(clean_df.columns))
print("Exact duplicates:", clean_df.duplicated().sum())
print("Missing dates:", clean_df["date"].isna().sum())
print("Missing organisation names:", clean_df["Name"].isna().sum())

clean_df.head()

Rows: 27108
Columns: 19
Exact duplicates: 0
Missing dates: 0
Missing organisation names: 0


,date,Name,Type 1 Departments - Major A&E,Type 2 Departments - Single Specialty,Type 3 Departments - Other A&E/Minor Injury Unit,Total attendances,Type 1 Departments - 4 hours to decision,Type 2 Departments - 4 hours to decision,Type 3 Departments - 4 hours to decision,Percentage in 4 hours or less (all),Emergency Admissions via Type 1 A&E in 4 hours,Emergency Admissions via Type 2 A&E in 4 hours,Emergency Admissions via Type 3 and 4 A&E in 4 hours,Other Emergency admissions (i.e not via A&E),Number of patients spending >12 hours from decision to admit to admission,month,year,lat,lon
0,2010-11-01,Aintree University Hospitals NHS Foundation Trust,4622.0,0.0,0.0,4622.0,7.0,0.0,0.0,2.995435,1406.0,0.0,0.0,530.0,0.0,11,2010,53.461606,-2.943427
1,2010-11-01,Airedale NHS Trust,3965.0,0.0,0.0,3965.0,65.0,0.0,0.0,3.934107,892.0,0.0,0.0,568.0,0.0,11,2010,NaN,NaN
2,2010-11-01,Alder Hey Children’S NHS Foundation Trust,4541.0,0.0,0.0,4541.0,71.0,0.0,0.0,3.937936,1825.0,0.0,0.0,270.0,0.0,11,2010,50.183000,-5.416000
3,2010-11-01,Ashford And St Peter'S Hospitals NHS Trust,7010.0,0.0,769.0,7779.0,392.0,0.0,0.0,3.797332,1902.0,0.0,0.0,163.0,0.0,11,2010,51.655000,-0.395694
4,2010-11-01,"Ashton, Leigh And Wigan Primary Care Trust",0.0,0.0,4297.0,4297.0,0.0,0.0,0.0,4.000000,0.0,0.0,0.0,0.0,0.0,11,2010,51.272000,0.529000
